[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/aimldstejas/aibits-genai-notebooks/blob/main/ml/04-machine-learning/ml-interpretability.ipynb)

# Model Interpretability & Explainability

*AIBits Academy · Machine Learning End To End · Responsible ML · New*

"The model said reject" is not an acceptable answer to an RBI auditor, a rejected loan applicant, or your own debugging process. This chapter covers explaining any model's individual predictions, not just its overall behaviour.

**How to use this notebook:** run the cells top to bottom (Runtime → Run all). Each code cell is the same code you saw on the course page, so you can compare your output with the lesson. The graded exercises are at the end; try them before opening the solutions.

*Interactive animations and quiz cards stay on the course page.*

## Setup

In [ ]:
# Packages that Colab does not ship by default (a no-op if already installed)
%pip install -q shap

## Global vs Local Explanations

|  | Global Explanation | Local Explanation |
|---|---|---|
| Question answered | "What does the model generally rely on, across all predictions?" | "Why did the model predict THIS for THIS specific person?" |
| Example | Random Forest feature_importances_ (already covered) | SHAP values for one rejected loan applicant |
| Use case | Model debugging, feature selection, high-level reporting | Individual adverse-action notices, case-by-case audits |

Global feature importance was already covered on the Random Forest page, along with its limitations (bias toward high-cardinality features, correlated-feature splitting). This chapter focuses on the harder, increasingly essential problem: explaining *individual* predictions, for *any* model type — not just trees.

## SHAP — Shapley Additive Explanations

SHAP borrows a concept from cooperative game theory: treat each feature as a "player" contributing to a "payout" (the prediction), and fairly distribute credit for the prediction among all features using Shapley values — the only attribution method satisfying a specific set of fairness axioms (efficiency, symmetry, additivity).

$$\text{prediction} = \text{base\_value} + \sum_j \mathrm{SHAP\_value}(\text{feature}_j) \qquad \text{(SHAP values sum exactly to the prediction)}$$

The base_value is the model's average prediction across the whole training set; each feature's SHAP value is how much that specific feature's value pushed *this* prediction above or below that baseline.

## Code — Explaining One Loan Rejection with SHAP

In [ ]:
import numpy as np
import shap
from sklearn.ensemble import RandomForestClassifier

# HDFC loan approval model — reuse the feature set from the Decision Trees chapter
np.random.seed(0)
n = 800
income = np.random.uniform(15,75,n); cibil = np.random.uniform(540,790,n)
emi_ratio = np.random.uniform(0.1,0.6,n)
X = np.column_stack([income, cibil, emi_ratio])
y = (((income>40)&(cibil>660))|(income>55)).astype(int)

clf = RandomForestClassifier(n_estimators=200, max_depth=6, random_state=42).fit(X, y)

explainer = shap.TreeExplainer(clf)
# One specific rejected applicant: low income, decent CIBIL, high EMI ratio
applicant = np.array([[22.0, 705.0, 0.52]])
sv_raw = explainer.shap_values(applicant)
shap_values = sv_raw[1] if isinstance(sv_raw, list) else sv_raw[:, :, 1]   # class 1 = approved (works for old and new SHAP)

feat_names = ['income_lakhs','cibil_score','emi_ratio']
for name, val, sv in zip(feat_names, applicant[0], shap_values[0]):
    direction = "pushed TOWARD approval" if sv>0 else "pushed TOWARD rejection"
    print(f"  {name}={val:6.2f}   SHAP={sv:+.3f}  ({direction})")
print(f"\nBase rate (avg approval prob): {explainer.expected_value[1]:.3f}")
print(f"This applicant's predicted probability: {clf.predict_proba(applicant)[0][1]:.3f}")

This is a genuine, individually-verifiable adverse-action explanation: "your application was scored below average primarily because of income (−0.284) and EMI ratio (−0.156), only partially offset by a good CIBIL score (+0.061)" — exactly the kind of specific, auditable reasoning a regulator or rejected applicant is entitled to, and something raw feature_importances_ (a single global ranking) cannot provide for an individual case.

## LIME — Local Interpretable Model-Agnostic Explanations

A different approach to the same local-explanation goal: perturb the input slightly (generate many nearby synthetic samples), see how the model's prediction changes, then fit a simple, interpretable model (usually linear) to *locally* approximate the complex model's behaviour just around that one point.

> **💡 SHAP vs LIME**
>
> SHAP has a stronger theoretical foundation (Shapley values are provably the unique fair attribution under standard game-theory axioms) and `TreeExplainer` is exact and fast for tree-based models specifically. LIME is more model-agnostic in practice (works identically for any black-box model, including deep networks and even non-differentiable pipelines) but its local linear approximation quality can vary depending on the perturbation sampling, and it lacks SHAP's additivity guarantee (LIME's attributions don't have to sum exactly to the prediction).

## Partial Dependence Plots (PDP)

Answers a different, global question: "holding all other features at their observed values, how does the prediction change as one feature varies?" — useful for understanding a feature's overall shape of effect (linear? threshold? U-shaped?) across the whole dataset, complementing SHAP's per-prediction view.

In [ ]:
from sklearn.inspection import partial_dependence

pd_result = partial_dependence(clf, X, features=[0], grid_resolution=10)  # income_lakhs
for inc, avg_pred in zip(pd_result['grid_values'][0], pd_result['average'][0]):
    print(f"  income=₹{inc:5.1f}L   avg. predicted approval prob={avg_pred:.3f}")

## Choosing a Tool

| Need | Tool |
|---|---|
| Explain one individual prediction, tree-based model | SHAP TreeExplainer (fast, exact) |
| Explain one individual prediction, any model type (including deep nets) | SHAP KernelExplainer, or LIME |
| Understand a feature's overall effect shape across the dataset | Partial Dependence Plot |
| Rank features by overall importance | SHAP global summary plot (mean \|SHAP value\|) — more reliable than raw tree feature_importances_ |

## The Question SHAP Can't Answer: Correlation vs. Causality

Every explainability tool on this page — SHAP, LIME, PDP, feature importance — tells you what the model *relies on*. None of them tell you whether that reliance reflects a genuine causal mechanism or a spurious correlation the model happened to find useful for prediction. This distinction matters enormously once explanations are used to justify real-world action, not just describe model behaviour.

> **⚠ A High SHAP Value Is Not Evidence of Causation**
>
> Suppose a Swiggy churn model gives "number of customer support tickets filed" a large positive SHAP value toward predicting churn. It's tempting to conclude "support tickets cause churn — reduce ticket volume to retain customers." But the more likely underlying reality is reversed causation: customers who are *already becoming dissatisfied* (for reasons the model never observed — bad experiences, competitor promotions) both file more tickets *and* churn more. Filing a ticket is a symptom correlated with an unobserved cause, not the cause itself. Acting on the SHAP explanation directly (e.g., discouraging customers from filing tickets) would actively make things worse, not better.

Formally, a feature x can be associated with outcome y through several distinct causal structures that SHAP cannot distinguish between just from its value:

| Structure | What it means | Acting on x |
|---|---|---|
| x → y (direct cause) | x genuinely causes y | Correctly changes y |
| y → x (reverse causation) | y actually causes x (as in the support-ticket example) | Does nothing to y, or backfires |
| z → x, z → y (confounding) | An unobserved z causes both — x and y are correlated but neither causes the other | No effect on y at all |

A lightweight first check, without a full causal-inference framework: does a plausible mechanism exist for x → y specifically, or only for the reverse/confounded stories? For anything higher-stakes — pricing decisions, policy interventions, medical treatment — dedicated causal inference methods (randomised experiments/A-B tests where feasible, or observational techniques like propensity score matching and instrumental variables where they aren't) are the rigorous way to distinguish these structures; SHAP and friends were never designed to answer that question and shouldn't be stretched to.

## ⚠ Advanced: From Recognising Confounding to Actually Estimating Causal Effects

The table above tells you confounding is *possible*. It doesn't tell you how to actually estimate the true causal effect once you suspect it. That's the job of causal inference proper — a full field in its own right, briefly introduced here at an intuitive level; the underlying statistical theory (high-dimensional inference, debiased/double machine learning) is substantial enough to warrant its own dedicated future course.

### ⚠ Potential Outcomes & the Average Treatment Effect (ATE)

For any one customer, there are two *potential* outcomes: Y(1), what would happen if they received a discount coupon, and Y(0), what would happen if they didn't. The uncomfortable fact underlying all of causal inference is that we only ever observe **one** of these two per customer — a customer either gets the coupon or doesn't, never both — so the individual causal effect Y(1)−Y(0) is fundamentally unobservable for any single unit. What *can* be estimated is the average across many units:

$$\mathrm{ATE} = E[Y(1) - Y(0)] \qquad \text{(estimated by comparing average outcomes across a treated group and a comparable control group)}$$

The entire causal-inference toolkit — RCTs, propensity matching, everything below — exists to answer one question: how do you construct a treated group and a control group that are genuinely *comparable*, so the difference in their average outcomes can be attributed to the treatment itself rather than to pre-existing differences between the groups?

### ⚠ Randomized Controlled Trials — Why Randomisation Works

An RCT (the same idea as the A/B testing already covered on the Recommender Systems page) randomly assigns units to treatment or control. This is a far stronger design than it might first appear: random assignment makes the two groups statistically identical, *on average*, across every characteristic — not just the ones you happened to measure, but every unobserved confounder too, including ones you'd never think to collect. This is precisely why an RCT can support a causal claim that pure observational analysis cannot.

|  | Observational Data | Randomized Controlled Trial |
|---|---|---|
| Who gets treated | Determined by real-world factors — often correlated with the outcome itself | Determined purely by a coin flip, unrelated to any customer characteristic |
| Confounding | A constant threat — treated and untreated groups likely differ in many ways besides treatment | Eliminated by design, in expectation — groups are balanced on both observed and unobserved factors |
| Limitation | Cheap, uses existing data | Often expensive, sometimes unethical or impossible (can't randomly assign smoking, or randomly assign who gets a heart attack) |

### ⚠ Propensity Score Matching — Approximating Randomisation Observationally

When an RCT isn't feasible — you can't ethically randomise who receives a price increase, or who develops a medical condition — the next-best option works with observational data instead. The **propensity score** is the probability of receiving treatment given observed characteristics, P(treatment=1 | X), typically estimated with exactly the Logistic Regression covered earlier in this course, using the observed confounders as predictors of treatment assignment rather than of the outcome.

Customers are then matched into pairs with similar propensity scores — one treated, one untreated — so that, *within each matched pair*, treatment assignment looks approximately as good as random, at least with respect to the confounders that went into the model. Comparing outcomes within these matched pairs approximates what a randomised comparison would have shown.

> **⚠ Matching Only Fixes What You Measured**
>
> Propensity score matching balances the groups on the confounders included in X — it does nothing whatsoever for confounders that were never observed or measured in the first place. If an important confounder is missing from the data (as in the earlier Swiggy support-ticket example, where the true driver of both tickets and churn was never captured), propensity matching will confidently produce a "balanced" comparison that is still biased. This is the single biggest practical limitation of every observational causal method, and the reason RCTs remain the gold standard whenever they're actually feasible.

### ⚠ Causal Diagrams (DAGs) & the Collider-Bias Trap

A causal diagram draws arrows representing believed cause-and-effect structure, and is used to decide which variables are safe to "control for" (condition on) and which are not. The confounder row in the table above — z → x, z → y — is exactly the case where conditioning on z is *correct and necessary*: it blocks the spurious backdoor path between x and y. But not every variable behaves like a confounder, and treating all "extra variables" as automatically safe to control for is a genuine trap.

A **collider** is a variable caused by *both* x and y (x → z ← y, arrows pointing *into* z rather than out of it). Conditioning on a collider does the opposite of conditioning on a confounder — it *creates* a spurious association between x and y where none existed before, rather than removing one.

> **🔀 A Concrete Collider Example**
>
> Suppose "Premium Subscription" status at Swiggy is caused both by high spending *and* by high satisfaction — two independent causes of the same downstream variable. If an analyst restricts their churn analysis to only premium subscribers (i.e., conditions on subscription status), a spurious negative association between spending and satisfaction can appear within that subgroup, purely as a statistical artefact of the selection — even if spending and satisfaction are genuinely unrelated to each other in the full population. This is why "just control for more variables" is not a universally safe instinct in causal analysis — whether a variable is a confounder (control for it) or a collider (leave it alone) depends entirely on the causal structure, which a diagram makes explicit and a correlation matrix alone cannot reveal.

### ❓ Conceptual Q&A

---
## Graded exercises

Each exercise has a **starter cell** you complete and a **check cell** that prints ✅ or ❌. The solution is folded away underneath — try first.

In [ ]:
# --- self-check helper (used by the exercises) ---------------------------------------------
def check(name, ok):
    print(("\u2705 " if ok else "\u274c ") + name)


### Exercise 1 · Easy · Permutation importance

Only column 0 drives the target. Compute `permutation_importance(model, X, y, n_repeats=10, random_state=0)` and store the index of the most important feature in `top` and the array of mean importances in `imps`.

In [ ]:
import numpy as np
from sklearn.ensemble import RandomForestClassifier
from sklearn.inspection import permutation_importance
rng = np.random.default_rng(0)
Xp = rng.normal(size=(600, 4))
yp = (Xp[:, 0] + 0.3 * rng.normal(size=600) > 0).astype(int)
model = RandomForestClassifier(n_estimators=100, random_state=0).fit(Xp, yp)
top = imps = None   # TODO


In [ ]:
try:
    check("column 0 is the top feature", top == 0)
    check("noise features are near zero", imps[1:].max() < 0.05)
except Exception as e:
    print("\u274c Not ready yet (" + type(e).__name__ + ") - complete the starter cell above, then run this again.")


<details><summary><b>Show solution</b></summary>

```python
import numpy as np
from sklearn.ensemble import RandomForestClassifier
from sklearn.inspection import permutation_importance
rng = np.random.default_rng(0)
Xp = rng.normal(size=(600, 4))
yp = (Xp[:, 0] + 0.3 * rng.normal(size=600) > 0).astype(int)
model = RandomForestClassifier(n_estimators=100, random_state=0).fit(Xp, yp)
imps = permutation_importance(model, Xp, yp, n_repeats=10, random_state=0).importances_mean
top = int(imps.argmax())

```

</details>

### Exercise 2 · Medium · Partial dependence

Using the lesson's `clf` and `X`, compute the partial dependence of approval probability on **income** (feature 0) with `grid_resolution=8` and store the averaged predictions (a 1-D array) in `pd_avg`. `rises` should say whether the last value is higher than the first.

In [ ]:
pd_avg = rises = None   # TODO


In [ ]:
try:
    check("eight grid points", len(pd_avg) == 8)
    check("approval rises with income", rises is True)
except Exception as e:
    print("\u274c Not ready yet (" + type(e).__name__ + ") - complete the starter cell above, then run this again.")


<details><summary><b>Show solution</b></summary>

```python
from sklearn.inspection import partial_dependence
pd_avg = partial_dependence(clf, X, features=[0], grid_resolution=8)["average"][0]
rises = bool(pd_avg[-1] > pd_avg[0])

```

</details>

### Exercise 3 · Stretch · SHAP values add up

SHAP's promise is *local accuracy*: base value + sum of the SHAP values = the model's prediction. For the lesson's `applicant`, store `reconstructed` (base + sum of class-1 SHAP values) and `predicted` (`clf.predict_proba(applicant)[0, 1]`), and check they agree.

In [ ]:
reconstructed = predicted = None   # TODO (reuse explainer, applicant, clf)


In [ ]:
try:
    check("values agree", abs(reconstructed - predicted) < 1e-6)
except Exception as e:
    print("\u274c Not ready yet (" + type(e).__name__ + ") - complete the starter cell above, then run this again.")


<details><summary><b>Show solution</b></summary>

```python
sv_raw = explainer.shap_values(applicant)
sv1 = sv_raw[1] if isinstance(sv_raw, list) else sv_raw[:, :, 1]
reconstructed = float(explainer.expected_value[1] + sv1[0].sum())
predicted = float(clf.predict_proba(applicant)[0, 1])

```

This additivity is what lets you show a rejected applicant exactly how each feature moved their score.

</details>

---
*Back to the course: **Machine Learning End To End → Model Interpretability & Explainability**.*